# VoxMeet · GPU transcript demo
1. Runtime → Change runtime type → T4 GPU.
2. Accept the Community-1 conditions at https://huggingface.co/pyannote/speaker-diarization-community-1 and add a read token as **HF_TOKEN** in Colab Secrets (enable notebook access).
3. Run setup. If it installs anything, restart the session, then continue with the processing cell.
4. In the local demo download **audio package**, upload that ZIP below. The audio is processed on Google Colab.
5. Download the result JSON and use **Apply GPU result** on the same local recording.

Uses the evaluated Community-1 **baseline**, not the fine-tuned checkpoint. Speaker numbers are recording-local labels, not verified identities. No training occurs here. Review the transcript before summarizing. GPU availability is controlled by Colab.


In [ ]:
import sys, subprocess, os
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError, distributions

wanted = {'numpy':'2.2.6', 'pyannote.audio':'4.0.7', 'faster-whisper':'1.2.1'}
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None

if all(installed(k) == v for k,v in wanted.items()):
    print('Libraries ready. Continue below.')
else:
    pins = dict(wanted)
    for name in ('torch','torchaudio','torchvision'):
        if installed(name): pins[name] = installed(name)
    for dist in distributions():
        name = dist.metadata.get('Name', '').lower().replace('_','-')
        if name.startswith('opentelemetry-'): pins[name] = dist.version
    constraints = Path('/content/voxmeet-demo-constraints.txt')
    constraints.write_text('\n'.join(f'{k}=={v}' for k,v in pins.items()))
    env = os.environ.copy()
    env.pop('PIP_CONSTRAINT', None)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(constraints),
                    *[f'{k}=={v}' for k,v in wanted.items()], 'jedi==0.19.2'],
                   env=env, check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'check'], env=env, check=True)
    print('Setup complete. Runtime → Restart session, then run the next cell.')


## Process one recording
Upload only the ZIP downloaded from the recording page. Progress is printed for ASR and diarization. The result is also saved locally in Colab; download it before ending the session.

In [ ]:
import gc, hashlib, io, json, time, zipfile
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from google.colab import files, userdata
from huggingface_hub import snapshot_download
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook

assert torch.cuda.is_available(), 'Enable T4 GPU first.'
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one VoxMeet ZIP.'
raw_zip = next(iter(uploaded.values()))
with zipfile.ZipFile(io.BytesIO(raw_zip)) as z:
    assert set(z.namelist()) == {'audio.wav', 'job.json'}, 'Not a VoxMeet audio package.'
    assert z.getinfo('audio.wav').file_size <= 120 * 1024 * 1024
    assert z.getinfo('job.json').file_size < 10000
    job = json.loads(z.read('job.json'))
    raw_audio = z.read('audio.wav')
assert job['schema'] == 'voxmeet-gpu-v1'
assert hashlib.sha256(raw_audio).hexdigest() == job['audio_sha256']
samples, sr = sf.read(io.BytesIO(raw_audio), dtype='float32', always_2d=True)
assert sr == 16000 and samples.shape[1] == 1 and np.isfinite(samples).all()
samples = samples[:, 0].copy()
assert 0 < len(samples) / sr <= 1801
revision = '3533c8cf8e369892e6b79ff1bf80f7b0286a54ee'
assert job['model_revision'] == revision
print(f'Audio ready: {len(samples)/sr:.2f}s. Recognizing English...', flush=True)
started = time.perf_counter()
# Run Whisper in an isolated process with CUDA 12 runtime libraries.
# PyTorch/pyannote keep their existing CUDA runtime in the notebook process.
import os, subprocess, sys
cuda_root = Path('/content/voxmeet_cuda12')
required_libs = ('libcublas.so.12', 'libcublasLt.so.12', 'libcudnn.so.9', 'libcudart.so.12', 'libnvrtc.so.12')
if not all(list(cuda_root.rglob(name)) for name in required_libs):
    print('Installing isolated CUDA 12 libraries for Whisper...', flush=True)
    install_env = os.environ.copy()
    install_env.pop('PIP_CONSTRAINT', None)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-deps',
                    '--target', str(cuda_root), 'nvidia-cublas-cu12',
                    'nvidia-cudnn-cu12>=9,<10', 'nvidia-cuda-runtime-cu12',
                    'nvidia-cuda-nvrtc-cu12'], env=install_env, check=True)
lib_dirs = sorted({str(p.parent) for p in cuda_root.rglob('*.so*')})
assert all(list(cuda_root.rglob(name)) for name in required_libs), 'CUDA 12 library installation is incomplete.'
asr_env = os.environ.copy()
asr_env['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + [asr_env.get('LD_LIBRARY_PATH', '')])
work = Path('/content/voxmeet_asr_job')
work.mkdir(exist_ok=True)
np.save(work / 'audio.npy', samples)
worker = work / 'recognize.py'
worker.write_text("""
import ctypes, json, sys
from pathlib import Path
import numpy as np
for name in ('libcublas.so.12', 'libcudnn.so.9'):
    ctypes.CDLL(name)
from faster_whisper import WhisperModel
work = Path(sys.argv[1])
samples = np.load(work / 'audio.npy')
model = WhisperModel('small.en', device='cuda', compute_type='float16')
parts, _ = model.transcribe(samples, language='en', beam_size=5,
                            word_timestamps=True, vad_filter=True,
                            condition_on_previous_text=False)
words = []
for index, part in enumerate(parts):
    words.extend(dict(start=float(w.start), end=float(w.end), word=w.word,
                      confidence=float(w.probability)) for w in (part.words or []))
    if index % 10 == 0: print(f'ASR: {part.end:.1f}/{len(samples)/16000:.1f}s', flush=True)
(work / 'words.json').write_text(json.dumps(words), encoding='utf-8')
""", encoding='utf-8')
subprocess.run([sys.executable, str(worker), str(work)], env=asr_env, check=True)
words = json.loads((work / 'words.json').read_text(encoding='utf-8'))

gc.collect(); torch.cuda.empty_cache()
assert words, 'No speech detected. Try clearer audio; no transcript will be replaced.'
print('Loading Community-1 baseline...', flush=True)
model_dir = snapshot_download('pyannote/speaker-diarization-community-1',
                              revision=revision, token=userdata.get('HF_TOKEN'))
pipeline = Pipeline.from_pretrained(model_dir).to(torch.device('cuda'))
with ProgressHook() as hook:
    output = pipeline({'waveform':torch.from_numpy(samples[None,:]), 'sample_rate':sr}, hook=hook)
turns = [dict(start=float(t.start), end=float(t.end), speaker=label)
         for t, _, label in output.speaker_diarization.itertracks(yield_label=True)]
from collections import defaultdict
def overlap(a, b):
    return max(0, min(a['end'], b['end']) - max(a['start'], b['start']))

def group_words(words):
    segments = []
    for word in words:
        text = word.get('punctuated_word', word.get('word', '')).strip()
        if not text:
            continue
        speaker = word.get('speaker')
        if speaker is not None:
            speaker = str(speaker)
        if (not segments or segments[-1]['speaker'] != speaker or
                word['start'] - segments[-1]['end'] > 2.5 or
                word['end'] - segments[-1]['start'] > 30):
            segments.append(dict(id=f's{len(segments)+1}', speaker=speaker,
                                 start=float(word['start']), end=float(word['end']), text=text, words=[]))
        else:
            segments[-1]['text'] += ' ' + text
            segments[-1]['end'] = max(segments[-1]['end'], float(word['end']))
        segments[-1]['words'].append(dict(text=text, start=float(word['start']), end=float(word['end']),
                                         confidence=word.get('confidence')))
    return segments

def assign_speakers(words, turns):
    for word in words:
        # Whisper sometimes returns point timestamps for short words. Assign
        # only when exactly one speaker is active at that point.
        if word['start'] == word['end']:
            speakers = {turn['speaker'] for turn in turns if turn['start'] <= word['start'] < turn['end']}
            word['speaker'] = next(iter(speakers)) if len(speakers) == 1 else None
            continue
        # A voice can have several adjacent intervals within one word. Compare
        # total covered time per voice, merging intervals to avoid double count.
        intervals = defaultdict(list)
        for turn in turns:
            if overlap(word, turn) > 0:
                intervals[turn['speaker']].append((max(word['start'], turn['start']),
                                                   min(word['end'], turn['end'])))
        scores = {}
        for speaker, spans in intervals.items():
            total, right = 0., float('-inf')
            for start, end in sorted(spans):
                total += max(0., end - max(start, right))
                right = max(right, end)
            scores[speaker] = total
        ranked = sorted(scores, key=scores.get, reverse=True)
        word['speaker'] = ranked[0] if ranked and (len(ranked) == 1 or
            scores[ranked[0]] - scores[ranked[1]] > 1e-6) else None
        word['ambiguous_speaker'] = len(ranked) > 1
    return words

utterances = group_words(assign_speakers(words, turns))
for index, utterance in enumerate(utterances, 1):
    utterance['id'] = f'U{index:04d}'
result = dict(job, utterances=utterances, words=words, diarization_turns=turns, provenance=dict(
    asr_model='Whisper small.en',diarization_model='Community-1 baseline',
    diarization_revision=revision), processing_seconds=round(time.perf_counter()-started,1))
result_path = Path('/content') / f"voxmeet-result-{job['recording_id']}.json"
result_path.write_text(json.dumps(result,ensure_ascii=False), encoding='utf-8')
print('Done:',len(utterances),'turns |',len({t['speaker'] for t in turns}),'speaker groups')
print('Unassigned words:',sum(w['speaker'] is None for w in words))
print('Review speaker labels in the local demo before summarizing.')
files.download(str(result_path))


In [ ]:
# If the download did not start, run this cell again.
from google.colab import files
files.download(str(result_path))
